In [2]:
import pandas as pd
import numpy as np

In [29]:

customers = pd.read_csv("data/customers.csv")
orders = pd.read_csv("data/sales_orders.csv")
returns = pd.read_csv("data/sales_returns.csv")

orders["order_date"] = pd.to_datetime(
    orders["order_date"], errors="coerce"
)

returns["return_date"] = pd.to_datetime(
    returns["return_date"], errors="coerce"
)

orders["paid_total_EGP"] = pd.to_numeric(
    orders["paid_total_EGP"], errors="coerce"
).fillna(0)

returns["refund_amount_EGP"] = pd.to_numeric(
    returns["refund_amount_EGP"], errors="coerce"
).fillna(0)

# Aggregate refunds at order level
refunds = (
    returns.groupby("sales_order_id")["refund_amount_EGP"]
    .sum()
    .reset_index()
)

orders = orders.merge(
    refunds,
    on="sales_order_id",
    how="left"
)

orders["refund_amount_EGP"] = (
    orders["refund_amount_EGP"].fillna(0)
)

orders["net_paid_EGP"] = (
    orders["paid_total_EGP"]
    - orders["refund_amount_EGP"]
).clip(lower=0)

# Attach customer information
orders = orders.merge(
    customers[
        ["customer_id", "governorate_ar",
         "age_band", "joined_date"]
    ],
    on="customer_id",
    how="left"
)


In [30]:
orders['order_date'].min()

Timestamp('2018-10-19 00:00:00')

In [31]:
snapshot_dates = pd.date_range(
    start="2018-10-19",
    end="2026-10-18",
    freq="QS"
)

In [32]:

def build_customer_snapshot(
    snapshot_date,
    customers,
    valid_orders,
    returns
):
    snapshot_date = pd.Timestamp(snapshot_date)

    history = valid_orders[
        (valid_orders["order_date"] <= snapshot_date)
    ].copy()

    # Only returns known at snapshot time
    historical_returns = returns[
        returns["return_date"] <= snapshot_date
    ].copy()

    # Aggregate historical refunds
    refund_totals = (
        historical_returns
        .groupby("sales_order_id")["refund_amount_EGP"]
        .sum()
    )

    history["refund_amount_EGP"] = (
        history["sales_order_id"]
        .map(refund_totals)
        .fillna(0)
    )

    history["net_paid_EGP"] = (
        history["paid_total_EGP"]
        - history["refund_amount_EGP"]
    ).clip(lower=0)

    rows = []

    for customer_id, group in history.groupby("customer_id"):

        group = group.sort_values("order_date")

        last_purchase = group["order_date"].max()
        first_purchase = group["order_date"].min()

        recency = (snapshot_date - last_purchase).days

        row = {
            "customer_id": customer_id,
            "snapshot_date": snapshot_date,
            "recency_days": recency,
            "customer_tenure_days": (
                snapshot_date - first_purchase
            ).days
        }

        for window in [30, 90, 180, 365]:

            start_date = (
                snapshot_date - pd.Timedelta(days=window)
            )

            window_data = group[
                group["order_date"] > start_date
            ]

            row[f"frequency_{window}d"] = (
                window_data["sales_order_id"].nunique()
            )

            row[f"monetary_{window}d"] = (
                window_data["net_paid_EGP"].sum()
            )

        row["avg_order_value_365d"] = (
            row["monetary_365d"] /
            max(row["frequency_365d"], 1)
        )

        order_dates = (
            group["order_date"]
            .drop_duplicates()
            .sort_values()
        )

        intervals = order_dates.diff().dt.days.dropna()

        row["avg_days_between_orders"] = (
            intervals.mean() if len(intervals) else np.nan
        )

        row["std_days_between_orders"] = (
            intervals.std() if len(intervals) > 1 else np.nan
        )

        recent = row["frequency_90d"]

        previous_start = snapshot_date - pd.Timedelta(days=180)
        previous_end = snapshot_date - pd.Timedelta(days=90)

        previous = group[
            (group["order_date"] > previous_start) &
            (group["order_date"] <= previous_end)
        ]["sales_order_id"].nunique()

        row["purchase_trend"] = (
            (recent + 1) / (previous + 1)
        )

        rows.append(row)

    return pd.DataFrame(rows)


In [33]:

def add_churn_target(
    snapshot_df,
    valid_orders,
    snapshot_date,
    horizon_days=180
):
    snapshot_date = pd.Timestamp(snapshot_date)

    future_end = (
        snapshot_date + pd.Timedelta(days=horizon_days)
    )

    # Future qualifying purchases
    future_orders = valid_orders[
        (valid_orders["order_date"] > snapshot_date) &
        (valid_orders["order_date"] <= future_end)
    ]

    future_frequency = (
        future_orders
        .groupby("customer_id")["sales_order_id"]
        .nunique()
    )

    result = snapshot_df.copy()

    result["future_frequency_180d"] = (
        result["customer_id"]
        .map(future_frequency)
        .fillna(0)
    )

    result["churn_label"] = (
        result["future_frequency_180d"] == 0
    ).astype(int)

    # Only customers active at snapshot
    result = result[
        result["recency_days"] < 180
    ].copy()

    return result


In [34]:
sales_orders = pd.read_csv(r'D:\HVIA\cotonil\data\sales_orders.csv')

In [35]:
sales_orders["order_date"] = pd.to_datetime(sales_orders["order_date"])

print(sales_orders["status"].value_counts(dropna=False))

status
Completed             448120
Delivered              81152
Partially returned     10728
Name: count, dtype: int64


In [36]:
sales_orders["order_date"] = pd.to_datetime(sales_orders["order_date"])

valid_orders = sales_orders[
    sales_orders["status"].isin(["Completed", "Delivered"])
].copy()

In [37]:
returns

,return_id,sales_order_id,sales_line_id,return_date,sku_id,return_quantity,refund_amount_EGP,reason,refund_method
0,RET0000001,SO00472785,SL000996848,2026-02-20,SKU000676,1,637.69,Size issue,Cash
1,RET0000002,SO00228394,SL000481462,2023-04-01,SKU000599,1,290.60,Defect claim,Cash
2,RET0000003,SO00307618,SL000648623,2024-04-15,SKU000533,3,931.99,Wrong item,Wallet
3,RET0000004,SO00374944,SL000790483,2025-02-12,SKU000278,1,215.40,Exchange,Wallet
4,RET0000005,SO00275488,SL000580915,2023-11-26,SKU000176,10,1040.96,Other,Wallet
...,...,...,...,...,...,...,...,...,...
28495,RET0028496,SO00347066,SL000731845,2024-10-22,SKU000335,1,137.01,Wrong item,Card reversal
28496,RET0028497,SO00465456,SL000981279,2026-02-07,SKU000621,1,432.22,Size issue,Cash
28497,RET0028498,SO00076021,SL000160263,2020-07-12,SKU000617,8,748.93,Defect claim,Wallet
28498,RET0028499,SO00124015,SL000261604,2021-07-10,SKU000547,1,39.19,Defect claim,Wallet


In [38]:
snapshot_dates = pd.date_range(
    start="2018-10-19",
    end="2026-10-18",
    freq="QS"
)

datasets = []

for snapshot_date in snapshot_dates:

    snapshot_df = build_customer_snapshot(
        snapshot_date=snapshot_date,
        customers=customers,
        valid_orders=valid_orders,
        returns=returns
    )

    if snapshot_df.empty:
        continue

    snapshot_df = add_churn_target(
        snapshot_df=snapshot_df,
        valid_orders=valid_orders,
        snapshot_date=snapshot_date,
        horizon_days=180
    )

    if not snapshot_df.empty:
        datasets.append(snapshot_df)

churn_dataset = pd.concat(
    datasets,
    ignore_index=True
)

churn_dataset = churn_dataset.sort_values(
    ["snapshot_date", "customer_id"]
).reset_index(drop=True)

print("Dataset Shape:", churn_dataset.shape)

churn_dataset.head()

Dataset Shape: (589966, 18)


,customer_id,snapshot_date,recency_days,customer_tenure_days,frequency_30d,monetary_30d,frequency_90d,monetary_90d,frequency_180d,monetary_180d,frequency_365d,monetary_365d,avg_order_value_365d,avg_days_between_orders,std_days_between_orders,purchase_trend,future_frequency_180d,churn_label
0,C0000001,2019-01-01,25,74,3,739.61,14,4157.02,14,4157.02,14,4157.02,296.930000,6.125000,7.623975,15.0,3.0,0
1,C0000002,2019-01-01,42,74,0,0.00,12,5351.38,12,5351.38,12,5351.38,445.948333,4.571429,4.540820,13.0,3.0,0
2,C0000003,2019-01-01,24,74,1,247.57,8,2650.66,8,2650.66,8,2650.66,331.332500,8.333333,8.140434,9.0,2.0,0
3,C0000004,2019-01-01,5,74,2,463.99,11,7588.42,11,7588.42,11,7588.42,689.856364,7.666667,8.602325,12.0,1.0,0
4,C0000005,2019-01-01,12,74,2,315.98,11,2729.90,11,2729.90,11,2729.90,248.172727,6.200000,6.442912,12.0,2.0,0


In [50]:
churn_dataset['customer_id'].nunique()

74027

In [39]:
churn_dataset.to_csv('churn_data.csv')

In [40]:
churn_dataset['churn_label'].value_counts()

churn_label
1    342065
0    247901
Name: count, dtype: int64

In [42]:
churn_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589966 entries, 0 to 589965
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   customer_id              589966 non-null  object        
 1   snapshot_date            589966 non-null  datetime64[ns]
 2   recency_days             589966 non-null  int64         
 3   customer_tenure_days     589966 non-null  int64         
 4   frequency_30d            589966 non-null  int64         
 5   monetary_30d             589966 non-null  float64       
 6   frequency_90d            589966 non-null  int64         
 7   monetary_90d             589966 non-null  float64       
 8   frequency_180d           589966 non-null  int64         
 9   monetary_180d            589966 non-null  float64       
 10  frequency_365d           589966 non-null  int64         
 11  monetary_365d            589966 non-null  float64       
 12  avg_order_value_

In [4]:
churn_dataset = pd.read_csv('churn_data.csv')

In [5]:
import tensorflow as tf 
from  tensorflow import keras 
from tensorflow.keras import layers 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report , accuracy_score ,precision_score , recall_score ,confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from scikeras.wrappers import KerasClassifier


In [20]:
def create_model(meta):
    model = keras.Sequential(
        [
            layers.Input(shape= (meta['n_features_in_'],)),
            layers.Dense(64,activation='relu'),
            layers.Dense(64,activation='relu'),
            layers.Dropout(.3),
            layers.Dense(32,activation='relu'),
            layers.Dense(32,activation='relu'),
            layers.Dropout(.2),
            layers.Dense(16,activation='relu'),

            layers.Dense(1,activation='sigmoid')
        ]
    )

    model.compile(
        optimizer = keras.optimizers.Adam(.01) ,
        loss = keras.losses.BinaryCrossentropy ,
        metrics = ['accuracy']
    )

    return model

In [ ]:
pip = Pipeline([
    ('imputer',SimpleImputer(strategy= 'constant',fill_value=0)),
    ('scaler',MinMaxScaler()),
    ('classifier',KerasClassifier(
        model=create_model,
        epochs=50,
        batch_size=32,
    
        verbose=0
    ))
    
])



In [24]:
churn_dataset = churn_dataset.sort_values(
    "snapshot_date"
).reset_index(drop=True)

dates = sorted(churn_dataset["snapshot_date"].unique())

train_end = int(len(dates) * 0.70)
val_end = int(len(dates) * 0.85)

train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]

train_df = churn_dataset[
    churn_dataset["snapshot_date"].isin(train_dates)
].copy()

val_df = churn_dataset[
    churn_dataset["snapshot_date"].isin(val_dates)
].copy()

test_df = churn_dataset[
    churn_dataset["snapshot_date"].isin(test_dates)
].copy()

In [25]:
target = "churn_label"

excluded_columns = [
    "customer_id",
    "snapshot_date",
    "churn_label",
    "future_frequency_180d"
]

feature_columns = [
    col for col in churn_dataset.columns
    if col not in excluded_columns
]

X_train = train_df[feature_columns]
y_train = train_df[target]

X_val = val_df[feature_columns]
y_val = val_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

In [26]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDate Ranges:")

print("Train:", train_df["snapshot_date"].min(),
      "to", train_df["snapshot_date"].max())

print("Validation:", val_df["snapshot_date"].min(),
      "to", val_df["snapshot_date"].max())

print("Test:", test_df["snapshot_date"].min(),
      "to", test_df["snapshot_date"].max())

Train: (310054, 15)
Validation: (129658, 15)
Test: (150254, 15)

Date Ranges:
Train: 2019-01-01 to 2024-04-01
Validation: 2024-07-01 to 2025-07-01
Test: 2025-10-01 to 2026-10-01


In [27]:
pip.fit(X_train, y_train)

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'constant'
,fill_value,0
,copy,True
,add_indicator,False
,keep_empty_features,False
,feature_range,"(0, ...)"
